In [1]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference, time_inference_torch

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import time
import os

In [3]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
RESULTS_CSV = "../../../2D_Results.csv"

In [2]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
target_col = 'Output'

# First set of features (Model explainability with log regression)
feature_cols_1 = ['TCD', 'PC', 'CN0', 'EC', 'PRN', 'LC', 'DO', 'PD' ,'CP', 'PIP']
X1 = df[feature_cols_1]
y = df[target_col]

# Second set of features (From Xgboost model with all features)
feature_cols_2 = ['TOW', 'PD', 'PRN', 'RX', 'TCD', 'DO', 'CP', 'CN0', 'PC', 'EC']
X2 = df[feature_cols_2]

# Split both sets with the same random state
X1_train, X1_test, y_train, y_test = train_test_split(
    X1, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)
X2_train, X2_test, _, _ = train_test_split(
    X2, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)  # y_train/y_test already made above

# Scale both sets
scaler1 = StandardScaler()
X1_train_scaled = scaler1.fit_transform(X1_train)
X1_test_scaled = scaler1.transform(X1_test)

scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.transform(X2_test)

In [5]:
results = []

In [ ]:
def add_results(model, model_name, feature_set, x_test, y_test, train_time, params, num_epochs):
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    print(f"{model_name} ({feature_set}) Accuracy: {accuracy * 100:.2f}%")
    print(f"Classification report ({model_name}, {feature_set}):")
    print(classification_report(y_test, y_pred, zero_division=0))
    _row = evaluate(
        y_test,
        y_pred,
        model_name=f"{model_name} ({feature_set})",
        dataset_name="GPS_Data_Simplified_2D_Feature_Map.xlsx",
        family="FeatureEngineering",
        labels=list(range(4)),
        class_names=["benign", "simple", "intermediate", "sophisticated"],
        y_proba=model.predict_proba(x_test) if hasattr(model, "predict_proba") else None,
        y_train=y_train,
        train_time_sec=train_time,
        num_epochs=num_epochs,
        parameters=str(params),
        seed=RANDOM_STATE,
        n_features=x_test.shape[1],
        n_train=len(y_train),
        test_set="real_heldout_20pct",
        timing=time_inference(model.predict, x_test),
    )
    print(summary(_row))
    results.append(_row)


In [7]:
# XGBoost on both sets
model1 = XGBClassifier(
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    eval_metric='mae',
    random_state=RANDOM_STATE,
)
start_time_1 = time.time()
model1.fit(X1_train_scaled, y_train)
train_time_1 = time.time() - start_time_1
add_results(model1, "XGBoost", "Explainability Features", X1_test_scaled, y_test, train_time_1, model1.get_params(), "N/A")

model2 = XGBClassifier(
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    eval_metric='mae',
    random_state=RANDOM_STATE,
)
start_time_2 = time.time()
model2.fit(X2_train_scaled, y_train)
train_time_2 = time.time() - start_time_2
add_results(model2, "XGBoost", "Xgboost Features", X2_test_scaled, y_test, train_time_2, model2.get_params(), "N/A")

XGBoost (Explainability Features) Accuracy: 93.14%
Classification report (XGBoost, Explainability Features):
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    318260
           1       0.81      0.81      0.81     29166
           2       0.73      0.70      0.72     35386
           3       0.95      0.94      0.94     25612

    accuracy                           0.93    408424
   macro avg       0.86      0.86      0.86    408424
weighted avg       0.93      0.93      0.93    408424

XGBoost (Xgboost Features) Accuracy: 93.32%
Classification report (XGBoost, Xgboost Features):
              precision    recall  f1-score   support

           0       0.96      0.97      0.97    318260
           1       0.82      0.82      0.82     29166
           2       0.73      0.71      0.72     35386
           3       0.95      0.94      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.87      0.86  

In [7]:
# MLPClassifier on both sets
mlp1 = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),
    activation='tanh',
    solver='adam',
    random_state=RANDOM_STATE,
    learning_rate='adaptive',
    early_stopping=True,
)
start_time_mlp1 = time.time()
mlp1.fit(X1_train_scaled, y_train)
train_time_mlp1 = time.time() - start_time_mlp1
add_results(mlp1, "MLP Classifier", "Explainability Features", X1_test_scaled, y_test, train_time_mlp1, mlp1.get_params(), mlp1.n_iter_)

mlp2 = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),
    activation='tanh',
    solver='adam',
    random_state=RANDOM_STATE,
    learning_rate='adaptive',
    early_stopping=True,
)
start_time_mlp2 = time.time()
mlp2.fit(X2_train_scaled, y_train)
train_time_mlp2 = time.time() - start_time_mlp2
add_results(mlp2, "MLP Classifier", "Xgboost Features", X2_test_scaled, y_test, train_time_mlp2, mlp2.get_params(), mlp2.n_iter_)

MLP Classifier (Explainability Features) Accuracy: 92.26%
Classification report (MLP Classifier, Explainability Features):
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    318260
           1       0.76      0.81      0.78     29166
           2       0.73      0.70      0.71     35386
           3       0.96      0.85      0.90     25612

    accuracy                           0.92    408424
   macro avg       0.85      0.83      0.84    408424
weighted avg       0.92      0.92      0.92    408424

MLP Classifier (Xgboost Features) Accuracy: 92.75%
Classification report (MLP Classifier, Xgboost Features):
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    318260
           1       0.78      0.79      0.79     29166
           2       0.75      0.71      0.73     35386
           3       0.94      0.91      0.92     25612

    accuracy                           0.93    408424
   macr

In [9]:
# Logistic Regression on both sets
logreg1 = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE,
)
start_time_logreg1 = time.time()
logreg1.fit(X1_train_scaled, y_train)
train_time_logreg1 = time.time() - start_time_logreg1
add_results(logreg1, "Logistic Regression", "Explainability Features", X1_test_scaled, y_test, train_time_logreg1, logreg1.get_params(), "N/A")

logreg2 = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE,
)
start_time_logreg2 = time.time()
logreg2.fit(X2_train_scaled, y_train)
train_time_logreg2 = time.time() - start_time_logreg2
add_results(logreg2, "Logistic Regression", "Xgboost Features", X2_test_scaled, y_test, train_time_logreg2, logreg2.get_params(), "N/A")

Logistic Regression (Explainability Features) Accuracy: 77.87%
Classification report (Logistic Regression, Explainability Features):
              precision    recall  f1-score   support

           0       0.78      1.00      0.88    318260
           1       0.00      0.00      0.00     29166
           2       0.05      0.00      0.00     35386
           3       0.30      0.00      0.00     25612

    accuracy                           0.78    408424
   macro avg       0.28      0.25      0.22    408424
weighted avg       0.63      0.78      0.68    408424

Logistic Regression (Xgboost Features) Accuracy: 77.89%
Classification report (Logistic Regression, Xgboost Features):
              precision    recall  f1-score   support

           0       0.78      1.00      0.88    318260
           1       0.00      0.00      0.00     29166
           2       0.08      0.00      0.00     35386
           3       0.00      0.00      0.00     25612

    accuracy                           0.

In [10]:
# Random Forest on both sets
rf1 = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
start_time_rf1 = time.time()
rf1.fit(X1_train_scaled, y_train)
train_time_rf1 = time.time() - start_time_rf1
add_results(rf1, "Random Forest", "Explainability Features", X1_test_scaled, y_test, train_time_rf1, rf1.get_params(), "N/A")

rf2 = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
start_time_rf2 = time.time()
rf2.fit(X2_train_scaled, y_train)
train_time_rf2 = time.time() - start_time_rf2
add_results(rf2, "Random Forest", "Xgboost Features", X2_test_scaled, y_test, train_time_rf2, rf2.get_params(), "N/A")

Random Forest (Explainability Features) Accuracy: 92.93%
Classification report (Random Forest, Explainability Features):
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    318260
           1       0.81      0.81      0.81     29166
           2       0.71      0.70      0.71     35386
           3       0.95      0.94      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.86      0.85      0.86    408424
weighted avg       0.93      0.93      0.93    408424

Random Forest (Xgboost Features) Accuracy: 93.09%
Classification report (Random Forest, Xgboost Features):
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    318260
           1       0.81      0.82      0.82     29166
           2       0.72      0.71      0.71     35386
           3       0.95      0.94      0.95     25612

    accuracy                           0.93    408424
   macro av

In [ ]:
# Save all results to CSV
append_row(results, RESULTS_CSV)
print(f"Results appended to {RESULTS_CSV}")